# 05 · Box detectors

**Goal:** a detector that puts boxes on the pneumonia opacities and scores as well as possible on the competition metric. In 06 its boxes are post-processed (gating by the classifier from 04, box size, threshold); in 07 it predicts the test images.

**Questions this notebook answers**
1. **Which architecture and which backbone?** Faster R-CNN, FCOS, RetinaNet and SSD on timm backbones, and 3 YOLO versions. (section 2)
2. **Does a stricter NMS help?** (section 3)
3. **Does a higher resolution or a larger YOLO help?** 256 vs 384 pixels, YOLO26s vs YOLO26m. (sections 4–5)
4. **Which detectors to ensemble?** Weighted boxes fusion of 1–5 of the best models on fold 0, then the chosen trio on all 5 folds. (sections 6–8)
5. **Does horizontal-flip TTA help?** (section 9)

Section 10 sums up the answers and the detectors chosen for 06.

Needs `patients.csv` and `boxes.csv` from `scripts/prepare_data.py`, and the detector runs in `outputs/detection/` from `scripts/train_detector.py` (torchvision), `scripts/train_yolo.py` (YOLO, on the dataset from `scripts/make_yolo_dataset.py`) and `scripts/predict_flip.py` (flip TTA).

In [1]:
import itertools
import json
import warnings

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import f1_score, roc_auc_score

from rsna.boxes import nms, rank_scores, wbf
from rsna.metrics import score_images
from rsna.paths import load_paths

pd.set_option("display.width", 200)
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 100)
warnings.filterwarnings("ignore", message="A worker stopped")  # joblib: harmless worker restarts

paths = load_paths()
DET = paths["outputs_dir"] / "detection"
ARCHS = {"fasterrcnn": "Faster R-CNN", "fcos": "FCOS", "retinanet": "RetinaNet", "ssd": "SSD"}
BACKBONES = {"convnext_tiny": "ConvNeXt-Tiny", "mobilevitv2_150": "MobileViTv2-1.5", "densenet121": "DenseNet-121",
             "tf_efficientnetv2_s": "EfficientNetV2-S", "efficientnet_b2": "EfficientNet-B2"}
YOLOS = {"yolo26s": "YOLO26s", "yolo11s": "YOLO11s", "yolov8s": "YOLOv8s", "yolo26m": "YOLO26m"}

# run key: (architecture, backbone or YOLO model, input size)
runs = {}
for a in ["fasterrcnn", "fcos", "retinanet"]:
    for b in BACKBONES:
        runs[(ARCHS[a], BACKBONES[b], 256)] = DET / "grid_256" / f"{a}_{b}_f0_s0"
for b in ["convnext_tiny", "mobilevitv2_150", "densenet121"]:
    runs[("SSD", BACKBONES[b], 256)] = DET / "ssd_256" / f"ssd_{b}_f0_s0"
for m in ["yolo26s", "yolo11s", "yolov8s"]:
    runs[("YOLO", YOLOS[m], 256)] = DET / "yolo_256" / f"{m}_f0_s0"
for a, b in [("fcos", "mobilevitv2_150"), ("fasterrcnn", "mobilevitv2_150"), ("fasterrcnn", "convnext_tiny")]:
    runs[(ARCHS[a], BACKBONES[b], 384)] = DET / "res_384" / f"{a}_{b}_f0_s0"
for m in ["yolo26s", "yolo26m"]:
    runs[("YOLO", YOLOS[m], 384)] = DET / "res_384" / f"{m}_f0_s0"
preds = {k: pd.read_csv(d / "val_preds.csv") for k, d in runs.items()}

patients = pd.read_csv(paths["processed_data_dir"] / "patients.csv")
true = pd.read_csv(paths["processed_data_dir"] / "boxes.csv")
val = patients[patients["fold"] == 0].reset_index(drop=True)
ids = val["patient_id"].to_numpy()
positive = val["target"].to_numpy() == 1
print(len(runs), "runs,", len(ids), "validation images,", positive.sum(), "with boxes")

26 runs, 5337 validation images, 1202 with boxes


## 1. Setup

- **Data:** fold 0: 21,347 training and 5,337 validation images (also those without boxes), at 256 × 256 unless noted.
- **torchvision detectors** (`scripts/train_detector.py`): Faster R-CNN, RetinaNet and FCOS on a pretrained timm backbone with an FPN; SSD on the same backbones without an FPN.
- **YOLO** (`scripts/train_yolo.py`, Ultralytics): COCO-pretrained YOLO26s, YOLO11s, YOLOv8s.
- **Evaluation:** the competition metric on the 5,337 validation images at the best box threshold. AUC and F1 use the image's highest box score (0 without boxes), F1 at that best threshold.
- **Ensembles:** weighted boxes fusion (WBF) of several detectors' boxes, sections 6-8.

In [2]:
THRESHOLDS = np.round(np.r_[np.arange(0.02, 0.10, 0.01), np.arange(0.10, 0.91, 0.05)], 2)
IOUS = ["own", 0.5, 0.4, 0.3, 0.2]  # "own" = the boxes as the model saved them


def nms_at(pred, iou):
    """Stricter NMS per image on the saved boxes (rsna.boxes.nms); "own" = unchanged."""
    return pred if iou == "own" else nms(pred, iou)


def threshold_scores(pred, image_ids):
    """Per-image scores (thresholds, images) at every box threshold; NaN = image skipped by the metric."""
    return np.stack([score_images(true, pred[pred["confidence"] >= t], image_ids).to_numpy()
                     for t in THRESHOLDS]).astype(np.float32)


def top_score(pred, image_ids):
    """Image score: the highest box score of the image (0 without boxes)."""
    return pred.groupby("patient_id")["confidence"].max().reindex(image_ids, fill_value=0).to_numpy()


# per-image scores of every run at every NMS IoU (8 processes)
tasks = [(k, iou) for k in preds for iou in IOUS]
image_scores = dict(zip(tasks, Parallel(n_jobs=8)(delayed(threshold_scores)(nms_at(preds[k], iou), ids) for k, iou in tasks)))
top_box = {k: top_score(p, ids) for k, p in preds.items()}


def result(k, iou):
    """Score at the best box threshold, with AUC and F1 (NMS keeps each image's top box, so AUC does not depend on it)."""
    mean = np.nanmean(image_scores[(k, iou)], axis=1)
    t = THRESHOLDS[mean.argmax()]
    return {"score": mean.max(), "threshold": t, "AUC": roc_auc_score(positive, top_box[k]),
            "F1": f1_score(positive, top_box[k] >= t)}

## 2. Detectors at 256 × 256

**Which architecture and which backbone?** Faster R-CNN, FCOS and RetinaNet, each on the 5 backbones, on fold 0. Competition score at the best box threshold, with each model's own NMS; means over the architectures and over the backbones:

In [3]:
table = pd.DataFrame({k[:2]: result(k, "own") for k in runs if k[2] == 256}).T.astype(float)
table.index.names = ["architecture", "model / backbone"]
pivot = table["score"].unstack(0)[["Faster R-CNN", "FCOS", "RetinaNet"]].loc[list(BACKBONES.values())].rename_axis("backbone")
pivot["mean"] = pivot.mean(axis=1)
pivot.loc["mean"] = pivot.mean()
pivot

architecture,Faster R-CNN,FCOS,RetinaNet,mean
backbone,,,,
ConvNeXt-Tiny,0.2388,0.2255,0.2185,0.2276
MobileViTv2-1.5,0.2399,0.2300,0.1965,0.2221
DenseNet-121,0.2322,0.2238,0.2073,0.2211
EfficientNetV2-S,0.2286,0.2085,0.1922,0.2098
EfficientNet-B2,0.2277,0.2085,0.1840,0.2067
mean,0.2334,0.2192,0.1997,0.2174


**Finding:** the two EfficientNets are the weakest backbones in all 3 architectures (mean 0.2098 and 0.2067, the others 0.2211–0.2276); MobileViTv2, ConvNeXt and DenseNet are close. Faster R-CNN is the best architecture with every backbone (mean 0.2334), then FCOS (0.2192) and RetinaNet (0.1997).

**All 21 runs at 256 × 256, fold 0:** the 15 above, SSD on 3 of the backbones and 3 YOLO versions of the same size (s). Competition score and its box threshold, AUC, and F1 at that threshold (own NMS), sorted by AUC:

In [4]:
table.sort_values("AUC", ascending=False)

score  threshold     AUC      F1
architecture model / backbone                                   
Faster R-CNN MobileViTv2-1.5   0.2399       0.75  0.8938  0.6448
RetinaNet    ConvNeXt-Tiny     0.2185       0.20  0.8929  0.6546
FCOS         MobileViTv2-1.5   0.2300       0.45  0.8921  0.6554
RetinaNet    MobileViTv2-1.5   0.1965       0.15  0.8905  0.6504
FCOS         ConvNeXt-Tiny     0.2255       0.45  0.8904  0.6563
Faster R-CNN ConvNeXt-Tiny     0.2388       0.70  0.8899  0.6563
             DenseNet-121      0.2322       0.60  0.8886  0.6480
RetinaNet    DenseNet-121      0.2073       0.20  0.8872  0.6229
YOLO         YOLO26s           0.1950       0.20  0.8869  0.5957
FCOS         DenseNet-121      0.2238       0.45  0.8869  0.6465
Faster R-CNN EfficientNet-B2   0.2277       0.60  0.8855  0.6473
FCOS         EfficientNet-B2   0.2085       0.45  0.8835  0.6258
YOLO         YOLOv8s           0.2028       0.15  0.8818  0.6104
             YOLO11s           0.1985       0.20  0.8811  0.6004
FCOS         EfficientNetV2-S  0.2085       0.45  0.8785  0.6304
RetinaNet    EfficientNetV2-S  0.1922       0.25  0.8746  0.5992
             EfficientNet-B2   0.1840       0.25  0.8744  0.5846
Faster R-CNN EfficientNetV2-S  0.2286       0.65  0.8677  0.6445
SSD          DenseNet-121      0.2073       0.45  0.8528  0.6034
             MobileViTv2-1.5   0.1970       0.45  0.8504  0.6029
             ConvNeXt-Tiny     0.2081       0.50  0.8495  0.5990

**Finding:** SSD is the worst architecture (AUC 0.850–0.853, every other run ≥ 0.868). The EfficientNets are not good either: after SSD they have the lowest AUCs (0.868–0.886). The best are Faster R-CNN, RetinaNet and FCOS on MobileViTv2-1.5, ConvNeXt-Tiny and DenseNet-121: the 8 highest AUCs (0.887–0.894). YOLO is in between (0.881–0.887).

## 3. Stricter NMS

The models keep boxes that overlap up to IoU 0.5–0.7, but the metric counts every extra box as a false positive. Here a stricter NMS is applied to the saved boxes afterwards. That is nearly the same as predicting with it: YOLO26s re-predicted with IoU 0.3 scores 0.2384, the saved boxes with NMS 0.3 afterwards 0.2380 (YOLO11s at 0.2: 0.2369 vs 0.2372). Only stricter NMS can be tried this way, and the boxes are at most 20 per image. YOLO26's end-to-end head without NMS (`nms=False`) scored 0.207.

Score at the best box threshold for each NMS IoU (*own* = the model's own NMS), the best IoU, its score and the improvement over the own NMS. *AUC (every IoU)*: the same for the own and every stricter NMS, because NMS keeps each image's top box, which gives the image score. Sorted by the best score:

In [5]:
at256 = [k for k in runs if k[2] == 256]
best_iou = {k: max(IOUS, key=lambda iou: result(k, iou)["score"]) for k in runs}
nms_table = pd.DataFrame({k[:2]: {**{iou: result(k, iou)["score"] for iou in IOUS}, "best IoU": best_iou[k],
                                  "score (best IoU)": result(k, best_iou[k])["score"],
                                  "score improvement": f'{result(k, best_iou[k])["score"] - result(k, "own")["score"]:+.4f}',
                                  "AUC (every IoU)": result(k, "own")["AUC"]}
                          for k in at256}).T
nms_table = nms_table.astype({col: float for col in nms_table.columns if col != "score improvement"})
nms_table.index.names = ["architecture", "model / backbone"]
nms_table.sort_values("score (best IoU)", ascending=False)

own     0.5     0.4     0.3     0.2  best IoU  score (best IoU) score improvement  AUC (every IoU)
architecture model / backbone                                                                                                       
Faster R-CNN MobileViTv2-1.5   0.2399  0.2399  0.2443  0.2437  0.2444       0.2            0.2444           +0.0045           0.8938
FCOS         MobileViTv2-1.5   0.2300  0.2360  0.2407  0.2418  0.2427       0.2            0.2427           +0.0127           0.8921
Faster R-CNN ConvNeXt-Tiny     0.2388  0.2388  0.2414  0.2422  0.2414       0.3            0.2422           +0.0034           0.8899
FCOS         ConvNeXt-Tiny     0.2255  0.2353  0.2390  0.2402  0.2401       0.3            0.2402           +0.0147           0.8904
Faster R-CNN DenseNet-121      0.2322  0.2322  0.2375  0.2384  0.2381       0.3            0.2384           +0.0063           0.8886
YOLO         YOLO26s           0.1950  0.2235  0.2358  0.2380  0.2375       0.3            0.2380           +0.0430           0.8869
             YOLO11s           0.1985  0.2268  0.2319  0.2367  0.2372       0.2            0.2372           +0.0387           0.8811
FCOS         DenseNet-121      0.2238  0.2322  0.2353  0.2358  0.2362       0.2            0.2362           +0.0125           0.8869
Faster R-CNN EfficientNet-B2   0.2277  0.2277  0.2337  0.2354  0.2360       0.2            0.2360           +0.0083           0.8855
             EfficientNetV2-S  0.2286  0.2286  0.2318  0.2320  0.2320       0.3            0.2320           +0.0035           0.8677
RetinaNet    MobileViTv2-1.5   0.1965  0.1965  0.2243  0.2310  0.2315       0.2            0.2315           +0.0350           0.8905
             DenseNet-121      0.2073  0.2073  0.2210  0.2293  0.2299       0.2            0.2299           +0.0226           0.8872
             ConvNeXt-Tiny     0.2185  0.2185  0.2237  0.2248  0.2247       0.3            0.2248           +0.0063           0.8929
YOLO         YOLOv8s           0.2028  0.2179  0.2209  0.2225  0.2232       0.2            0.2232           +0.0204           0.8818
FCOS         EfficientNet-B2   0.2085  0.2145  0.2177  0.2196  0.2205       0.2            0.2205           +0.0120           0.8835
             EfficientNetV2-S  0.2085  0.2151  0.2158  0.2169  0.2170       0.2            0.2170           +0.0085           0.8785
RetinaNet    EfficientNetV2-S  0.1922  0.1922  0.2055  0.2096  0.2105       0.2            0.2105           +0.0182           0.8746
SSD          ConvNeXt-Tiny     0.2081  0.2081  0.2081  0.2082  0.2082       0.3            0.2082           +0.0001           0.8495
             DenseNet-121      0.2073  0.2073  0.2075  0.2072  0.2072       0.4            0.2075           +0.0002           0.8528
RetinaNet    EfficientNet-B2   0.1840  0.1840  0.1979  0.2047  0.2065       0.2            0.2065           +0.0225           0.8744
SSD          MobileViTv2-1.5   0.1970  0.1970  0.1970  0.1969  0.1969       0.4            0.1970           +0.0000           0.8504

**Finding:**
- Stricter NMS helps every model except SSD. For all the others the best IoU is 0.2 or 0.3, and 0.2 and 0.3 differ by at most 0.002.
- The looser the model's own NMS, the larger the gain: YOLO (own IoU 0.7) +0.020 to +0.043, FCOS and RetinaNet (0.6) +0.006 to +0.035, Faster R-CNN (0.5) +0.003 to +0.008, SSD (0.45) about 0.
- The ranking changes: YOLO26s and YOLO11s move from near the bottom (0.195–0.199) to 6th and 7th (0.238 and 0.237), close to the best Faster R-CNN (0.244). The top 3 stay Faster R-CNN + MobileViTv2, FCOS + MobileViTv2 and Faster R-CNN + ConvNeXt.
- From here on, each model is used at its best NMS IoU.

## 4. 384 vs 256

The 4 best models at 256 (Faster R-CNN with MobileViTv2-1.5 and ConvNeXt-Tiny, FCOS with MobileViTv2-1.5, YOLO26s) trained again at 384 × 384, same settings. Score with the own NMS and at the best NMS IoU (section 3), with its threshold, AUC and F1:

In [6]:
at384 = [k for k in runs if k[2] == 384 and (k[0], k[1], 256) in runs]
rows = {}
for k in at384:
    for size in (256, 384):
        r = (k[0], k[1], size)
        rows[(*k[:2], size)] = {"own NMS": result(r, "own")["score"], "best NMS": result(r, best_iou[r])["score"],
                                "best IoU": best_iou[r], **{c: v for c, v in result(r, best_iou[r]).items() if c != "score"}}
pd.DataFrame(rows).T.rename_axis(["architecture", "model / backbone", "size"])

own NMS  best NMS  best IoU  threshold     AUC      F1
architecture model / backbone size                                                        
FCOS         MobileViTv2-1.5  256    0.2300    0.2427       0.2       0.45  0.8921  0.6554
                              384    0.2211    0.2416       0.2       0.45  0.8924  0.6600
Faster R-CNN MobileViTv2-1.5  256    0.2399    0.2444       0.2       0.70  0.8938  0.6558
                              384    0.2392    0.2420       0.2       0.75  0.8938  0.6537
             ConvNeXt-Tiny    256    0.2388    0.2422       0.3       0.70  0.8899  0.6563
                              384    0.2372    0.2418       0.3       0.65  0.8948  0.6654
YOLO         YOLO26s          256    0.1950    0.2380       0.3       0.10  0.8869  0.6544
                              384    0.2061    0.2312       0.2       0.15  0.8863  0.6272

**Paired bootstrap** on the 5,337 validation images: 2,000 resamples of the images; in each, the score of a run is the mean over the resampled images (images without true and predicted boxes are skipped, as in the metric). The intervals cover the choice of images only, not seed noise or the picking of NMS and threshold.

384 minus 256 for the same model, each at its best NMS, and both at NMS 0.3:

In [7]:
best = {k: image_scores[(k, best_iou[k])][np.nanmean(image_scores[(k, best_iou[k])], axis=1).argmax()] for k in runs}
SAMPLES = np.random.default_rng(0).integers(0, len(ids), (2000, len(ids)))


def interval(diff, diff_boot):
    low, high = np.percentile(diff_boot, [2.5, 97.5])
    return {"difference": diff, "95% interval": f"[{low:.4f}, {high:.4f}]"}


def diff(a, b, samples):
    """Score of per-image scores a minus b, with the bootstrap interval."""
    return interval(np.nanmean(a) - np.nanmean(b), np.nanmean(a[samples], axis=1) - np.nanmean(b[samples], axis=1))


def at_nms(k, iou):
    s = image_scores[(k, iou)]
    return s[np.nanmean(s, axis=1).argmax()]


rows = {}
for k in at384:
    k256 = (k[0], k[1], 256)
    name = k[1] if k[0] == "YOLO" else " + ".join(k[:2])
    rows[f"{name}: best NMS"] = diff(best[k], best[k256], SAMPLES)
    rows[f"{name}: NMS 0.3"] = diff(at_nms(k, 0.3), at_nms(k256, 0.3), SAMPLES)
pd.DataFrame(rows).T

,difference,95% interval
FCOS + MobileViTv2-1.5: best NMS,-0.0011,"[-0.0115, 0.0086]"
FCOS + MobileViTv2-1.5: NMS 0.3,-0.0004,"[-0.0106, 0.0092]"
Faster R-CNN + MobileViTv2-1.5: best NMS,-0.0025,"[-0.0123, 0.0074]"
Faster R-CNN + MobileViTv2-1.5: NMS 0.3,-0.0021,"[-0.0117, 0.0078]"
Faster R-CNN + ConvNeXt-Tiny: best NMS,-0.0004,"[-0.0107, 0.0101]"
Faster R-CNN + ConvNeXt-Tiny: NMS 0.3,-0.0004,"[-0.0107, 0.0101]"
YOLO26s: best NMS,-0.0068,"[-0.0177, 0.0048]"
YOLO26s: NMS 0.3,-0.0087,"[-0.0197, 0.0030]"


**Finding:** 384 × 384 does not help. At the best NMS all 4 models score slightly lower at 384 (−0.0004 to −0.0068), and at NMS 0.3 as well; every interval includes 0. AUC is about the same (Faster R-CNN + ConvNeXt +0.005, the others within ±0.001). The detectors stay at 256 × 256.

## 5. YOLO26s vs YOLO26m at 384 × 384

The medium YOLO26 (YOLO26m) trained at 384 only, compared with YOLO26s at 384:

In [8]:
yolo384 = [("YOLO", m, 384) for m in ["YOLO26s", "YOLO26m"]]
pd.DataFrame({k[1]: {"own NMS": result(k, "own")["score"], "best NMS": result(k, best_iou[k])["score"], "best IoU": best_iou[k],
                     **{c: v for c, v in result(k, best_iou[k]).items() if c != "score"}} for k in yolo384}).T     .rename_axis("model, 384 × 384")

,own NMS,best NMS,best IoU,threshold,AUC,F1
"model, 384 × 384",,,,,,
YOLO26s,0.2061,0.2312,0.2,0.15,0.8863,0.6272
YOLO26m,0.2039,0.2385,0.2,0.10,0.8877,0.6500


YOLO26m minus YOLO26s at their best NMS (IoU 0.2 for both), paired bootstrap as in section 4:

In [9]:
pd.DataFrame({"YOLO26m − YOLO26s": diff(best[yolo384[1]], best[yolo384[0]], SAMPLES)}).T

,difference,95% interval
YOLO26m − YOLO26s,0.0073,"[-0.0035, 0.0180]"


**Finding:** YOLO26m is +0.0073 over YOLO26s at 384 (interval −0.0035 to +0.0180, includes 0), and only level with YOLO26s at 256 (0.2385 vs 0.2380), with a similar AUC (0.8877 vs 0.8869). It is still tried as an ensemble candidate in section 6.

## 6. Which detectors to ensemble (fold 0)

Candidates: the 5 best models, each after its best stricter NMS (section 3). Their boxes are fused per image by **weighted boxes fusion** (`rsna.boxes.wbf`):

- **Scores:** raw, or rank-normalized (each box's percentile rank among its model's boxes), because YOLO's scores are much lower than the others'.
- **Matching:** the boxes of all models, sorted by weighted score (score × model weight), one by one: a box joins the fused box it overlaps most if their IoU > *WBF IoU*, otherwise it starts a new fused box.
- **Fused box:** coordinates = weighted-score mean of its boxes; score = weighted mean over the models of each model's top score in it (0 for a model without a box there), so each model votes once.
- **Tried per combination:** raw / rank × weights (equal, or one model × 2) × WBF IoU 0.2–0.5 × box threshold.
- **split-half:** the settings and threshold are picked on a random half of the images and scored on the other half (20 splits × both halves). This removes the optimism of picking them on the scored images (for single models: NMS and threshold), but not that of picking the best combination among many.

In [10]:
CANDIDATES = {"FRCNN-MViT": ("Faster R-CNN", "MobileViTv2-1.5", 256), "FRCNN-ConvNeXt": ("Faster R-CNN", "ConvNeXt-Tiny", 256),
              "FCOS-MViT": ("FCOS", "MobileViTv2-1.5", 256), "YOLO26s": ("YOLO", "YOLO26s", 256),
              "YOLO26m-384": ("YOLO", "YOLO26m", 384)}
WBF_IOUS = [0.2, 0.3, 0.4, 0.5]


def split_half(arr, n_splits=20):
    """arr (settings, thresholds, images): mean held-out score when setting and threshold are picked on the other half."""
    rng, held = np.random.default_rng(0), []
    for _ in range(n_splits):
        perm = rng.permutation(arr.shape[2])
        halves = perm[: len(perm) // 2], perm[len(perm) // 2:]
        for a, b in (halves, halves[::-1]):
            mean = np.nanmean(arr[:, :, a], axis=2)
            i, j = np.unravel_index(mean.argmax(), mean.shape)
            held.append(np.nanmean(arr[i, j, b]))
    return np.mean(held)


def summarize(settings, tops, positive_):
    """Best setting and threshold on all images, split-half, AUC / F1; also returns the best per-image scores."""
    names = list(settings)
    arr = np.stack([settings[n] for n in names])
    mean = np.nanmean(arr, axis=2)
    si, ti = np.unravel_index(mean.argmax(), mean.shape)
    t = THRESHOLDS[ti]
    row = {"score": mean[si, ti], "split-half": split_half(arr), "setting": names[si], "threshold": t,
           "AUC": roc_auc_score(positive_, tops[names[si]]), "F1": f1_score(positive_, tops[names[si]] >= t)}
    return row, arr[si, ti]


def fuse_and_score(model_preds, weights, iou, image_ids):
    fused = wbf(model_preds, weights, iou)
    return threshold_scores(fused, image_ids), top_score(fused, image_ids)


def search(inputs, image_ids, positive_):
    """inputs: {model: {"raw": boxes, "rank": boxes}}; WBF at every setting (8 processes), then summarize."""
    models = list(inputs)
    weight_sets = [(1,) * len(models)] + [tuple(2 if i == j else 1 for i in range(len(models))) for j in range(len(models))]
    settings = [(norm, w, iou) for norm in ("raw", "rank") for w in weight_sets for iou in WBF_IOUS]
    out = Parallel(n_jobs=8)(delayed(fuse_and_score)([inputs[m][norm] for m in models], w, iou, image_ids)
                             for norm, w, iou in settings)
    names = [f"{norm}, w={w}, IoU {iou}" for norm, w, iou in settings]
    return summarize(dict(zip(names, [o[0] for o in out])), dict(zip(names, [o[1] for o in out])), positive_)


def single(scores, top, positive_):
    """A single model: NMS and threshold picked as in section 3."""
    return summarize({f"NMS {iou}": scores[iou] for iou in IOUS}, {f"NMS {iou}": top for iou in IOUS}, positive_)


inputs0 = {}
for m, k in CANDIDATES.items():
    p = nms_at(preds[k], best_iou[k])
    inputs0[m] = {"raw": p, "rank": rank_scores(p)}
ens0, vec0 = {}, {}
for m, k in CANDIDATES.items():
    ens0[(m,)], vec0[(m,)] = single({iou: image_scores[(k, iou)] for iou in IOUS}, top_box[k], positive)
for n in (2, 3, 4, 5):
    for combo in itertools.combinations(CANDIDATES, n):
        ens0[combo], vec0[combo] = search({m: inputs0[m] for m in combo}, ids, positive)
table0 = pd.DataFrame({" + ".join(c): {"models": len(c), **r} for c, r in ens0.items()}).T.astype({"models": int})
print(len(ens0), "combinations")

31 combinations


The 3 best combinations of each size (weights in the order of the models):

In [11]:
table0.sort_values("score", ascending=False).groupby("models", sort=False).head(3).sort_values(
    ["models", "score"], ascending=[True, False])

,models,score,split-half,setting,threshold,AUC,F1
FRCNN-MViT,1,0.2444,0.2415,NMS 0.2,0.7,0.8938,0.6558
FCOS-MViT,1,0.2427,0.2427,NMS 0.2,0.45,0.8921,0.6554
FRCNN-ConvNeXt,1,0.2422,0.2404,NMS 0.3,0.7,0.8899,0.6563
FRCNN-MViT + FRCNN-ConvNeXt,2,0.2539,0.2512,"raw, w=(1, 1), IoU 0.3",0.65,0.8955,0.6657
FRCNN-ConvNeXt + YOLO26s,2,0.2539,0.2505,"raw, w=(1, 2), IoU 0.3",0.3,0.8934,0.6591
FRCNN-MViT + YOLO26s,2,0.2534,0.2486,"raw, w=(1, 2), IoU 0.4",0.3,0.8949,0.6602
FRCNN-ConvNeXt + FCOS-MViT + YOLO26s,3,0.2616,0.26,"rank, w=(1, 1, 1), IoU 0.3",0.8,0.8959,0.6672
FRCNN-MViT + FRCNN-ConvNeXt + YOLO26s,3,0.2568,0.2512,"rank, w=(1, 1, 1), IoU 0.3",0.75,0.8965,0.6644
FRCNN-MViT + FRCNN-ConvNeXt + YOLO26m-384,3,0.2567,0.2528,"raw, w=(1, 1, 2), IoU 0.3",0.4,0.897,0.6567
FRCNN-ConvNeXt + FCOS-MViT + YOLO26s + YOLO26m-384,4,0.2599,0.2545,"rank, w=(1, 1, 1, 1), IoU 0.3",0.75,0.8963,0.6618


**Leave one out:** the 5-model ensemble minus each 4-model ensemble without one of the models (bootstrap as in section 4):

In [12]:
all5 = tuple(CANDIDATES)
pd.DataFrame({m: {"without it": ens0[tuple(c for c in all5 if c != m)]["score"],
                  "split-half": ens0[tuple(c for c in all5 if c != m)]["split-half"],
                  **diff(vec0[all5], vec0[tuple(c for c in all5 if c != m)], SAMPLES)} for m in all5}).T \
    .rename_axis("left out").rename(columns={"difference": "5 models minus it"})

,without it,split-half,5 models minus it,95% interval
left out,,,,
FRCNN-MViT,0.2599,0.2545,0.001,"[-0.0044, 0.0066]"
FRCNN-ConvNeXt,0.2582,0.2538,0.0027,"[-0.0030, 0.0084]"
FCOS-MViT,0.2591,0.2541,0.0018,"[-0.0038, 0.0073]"
YOLO26s,0.2574,0.252,0.0035,"[-0.0017, 0.0088]"
YOLO26m-384,0.259,0.2523,0.0019,"[-0.0014, 0.0054]"


**All 10 trios**, minus the 5-model ensemble:

In [13]:
trios = sorted([c for c in ens0 if len(c) == 3], key=lambda c: ens0[c]["score"], reverse=True)
pd.DataFrame({" + ".join(c): {"score": ens0[c]["score"], "split-half": ens0[c]["split-half"], "AUC": ens0[c]["AUC"],
                              "F1": ens0[c]["F1"], **diff(vec0[c], vec0[all5], SAMPLES)} for c in trios}).T

,score,split-half,AUC,F1,difference,95% interval
FRCNN-ConvNeXt + FCOS-MViT + YOLO26s,0.2616,0.26,0.8959,0.6672,0.0007,"[-0.0048, 0.0062]"
FRCNN-MViT + FRCNN-ConvNeXt + YOLO26s,0.2568,0.2512,0.8965,0.6644,-0.0041,"[-0.0094, 0.0008]"
FRCNN-MViT + FRCNN-ConvNeXt + YOLO26m-384,0.2567,0.2528,0.897,0.6567,-0.0042,"[-0.0093, 0.0007]"
FRCNN-MViT + FRCNN-ConvNeXt + FCOS-MViT,0.2565,0.2539,0.895,0.6598,-0.0044,"[-0.0103, 0.0012]"
FRCNN-MViT + YOLO26s + YOLO26m-384,0.2553,0.2509,0.8959,0.6586,-0.0056,"[-0.0107, -0.0007]"
FRCNN-MViT + FCOS-MViT + YOLO26s,0.2553,0.2509,0.8972,0.6595,-0.0056,"[-0.0120, 0.0004]"
FRCNN-MViT + FCOS-MViT + YOLO26m-384,0.2552,0.2496,0.8972,0.6557,-0.0057,"[-0.0119, 0.0003]"
FRCNN-ConvNeXt + YOLO26s + YOLO26m-384,0.255,0.2507,0.8945,0.6576,-0.0059,"[-0.0119, 0.0002]"
FRCNN-ConvNeXt + FCOS-MViT + YOLO26m-384,0.2547,0.2485,0.8961,0.6507,-0.0062,"[-0.0130, 0.0006]"
FCOS-MViT + YOLO26s + YOLO26m-384,0.2533,0.2482,0.8955,0.6555,-0.0076,"[-0.0147, -0.0009]"


**Finding:**
- Up to 3 models, every extra model helps: best single model 0.2444, pair 0.2539, trio 0.2616. With 4 and 5 models the score does not rise further (0.2599, 0.2609).
- Leaving any one model out of the 5 changes the score by at most 0.0035 (every interval includes 0).
- The best trio, Faster R-CNN + ConvNeXt-Tiny, FCOS + MobileViTv2-1.5 and YOLO26s, is level with the 5 models (+0.0007, interval −0.0048 to +0.0062) and 0.004 or more above every other trio. It has one model of each family. It is chosen: 3 models to train on the other 4 folds instead of 5.

## 7. The chosen trio on all 5 folds

Faster R-CNN + ConvNeXt-Tiny, FCOS + MobileViTv2-1.5 and YOLO26s: one model of each family, and the best trio above. Folds 1–4 trained with the same settings as fold 0. Score per fold with each model's own NMS (at the best threshold of the fold):

In [14]:
CHOSEN = {"FRCNN-ConvNeXt": ("grid_256", "fasterrcnn_convnext_tiny"), "FCOS-MViT": ("grid_256", "fcos_mobilevitv2_150"),
          "YOLO26s": ("yolo_256", "yolo26s")}


def fold_dir(model, fold):
    experiment, run = CHOSEN[model]
    return DET / (experiment if fold == 0 else "cv_256") / f"{run}_f{fold}_s0"


cv = pd.DataFrame({(m, f): json.loads((fold_dir(m, f) / "summary.json").read_text()) for m in CHOSEN for f in range(5)}).T
cv = cv[["score", "threshold", "auc", "f1"]].astype(float)
cv.columns = ["score", "threshold", "AUC", "F1"]
cv.index.names = ["model", "fold"]
per_fold = cv["score"].unstack()
per_fold["mean"], per_fold["std"] = per_fold.mean(axis=1), per_fold.std(axis=1)
per_fold

fold,0,1,2,3,4,mean,std
model,,,,,,,
FCOS-MViT,0.2300,0.2244,0.2125,0.2218,0.2150,0.2207,0.0071
FRCNN-ConvNeXt,0.2388,0.2373,0.2211,0.2284,0.2274,0.2306,0.0074
YOLO26s,0.1950,0.1934,0.1875,0.1951,0.2009,0.1944,0.0048


In [15]:
cv

score  threshold     AUC      F1
model          fold                                   
FRCNN-ConvNeXt 0     0.2388       0.70  0.8899  0.6563
               1     0.2373       0.70  0.8936  0.6600
               2     0.2211       0.75  0.8898  0.6544
               3     0.2284       0.70  0.8955  0.6530
               4     0.2274       0.75  0.8937  0.6604
FCOS-MViT      0     0.2300       0.45  0.8921  0.6554
               1     0.2244       0.45  0.8906  0.6592
               2     0.2125       0.45  0.8887  0.6511
               3     0.2218       0.45  0.8930  0.6553
               4     0.2150       0.45  0.8947  0.6601
YOLO26s        0     0.1950       0.20  0.8869  0.5957
               1     0.1934       0.15  0.8912  0.6543
               2     0.1875       0.15  0.8818  0.6445
               3     0.1951       0.15  0.8858  0.6458
               4     0.2009       0.15  0.8877  0.6352

Out-of-fold (all 26,684 training images, each scored by the model that did not train on it), score by NMS IoU:

In [16]:
oof_ids = patients["patient_id"].to_numpy()
oof_pos = patients["target"].to_numpy() == 1
fold_of = patients["fold"].to_numpy()
oof = {m: pd.concat([pd.read_csv(fold_dir(m, f) / "val_preds.csv").assign(fold=f) for f in range(5)], ignore_index=True)
       for m in CHOSEN}
tasks = [(m, iou) for m in oof for iou in IOUS]
oof_scores = dict(zip(tasks, Parallel(n_jobs=8)(delayed(threshold_scores)(nms_at(oof[m], iou), oof_ids) for m, iou in tasks)))
oof_top = {m: top_score(p, oof_ids) for m, p in oof.items()}
pd.DataFrame({m: {iou: np.nanmean(oof_scores[(m, iou)], axis=1).max() for iou in IOUS} for m in oof}).T

,own,0.5,0.4,0.3,0.2
FRCNN-ConvNeXt,0.2291,0.2291,0.2329,0.2336,0.2333
FCOS-MViT,0.2207,0.2287,0.2313,0.2316,0.2318
YOLO26s,0.1940,0.2223,0.2278,0.2305,0.2311


**Finding:** fold 0 was one of the easier folds: Faster R-CNN and FCOS score 0.008–0.009 above their 5-fold mean (0.2306 and 0.2207), YOLO26s about its mean (0.1944); fold 2 is the lowest for all three. Out-of-fold, the best NMS IoU is again 0.2–0.3.

## 8. Ensembles on all 5 folds

The 3 pairs and the trio, same search as in section 6, on the out-of-fold images. Each model after its best out-of-fold NMS; rank normalization within each fold's model (each has its own score scale).

In [17]:
oof_best_iou = {m: max(IOUS, key=lambda iou: np.nanmean(oof_scores[(m, iou)], axis=1).max()) for m in oof}
inputs = {}
for m, p in oof.items():
    q = nms_at(p, oof_best_iou[m])
    inputs[m] = {"raw": q, "rank": rank_scores(q, by="fold")}
ens, vec = {}, {}
for m in oof:
    ens[(m,)], vec[(m,)] = single({iou: oof_scores[(m, iou)] for iou in IOUS}, oof_top[m], oof_pos)
for n in (2, 3):
    for combo in itertools.combinations(oof, n):
        ens[combo], vec[combo] = search({m: inputs[m] for m in combo}, oof_ids, oof_pos)
pd.DataFrame({" + ".join(c): {**r, **{f"fold {f}": np.nanmean(v[fold_of == f]) for f in range(5)}}
              for c, r in ens.items() for v in [vec[c]]}).T.sort_values("score", ascending=False)

,score,split-half,setting,threshold,AUC,F1,fold 0,fold 1,fold 2,fold 3,fold 4
FRCNN-ConvNeXt + FCOS-MViT + YOLO26s,0.251,0.25,"rank, w=(1, 1, 1), IoU 0.3",0.8,0.8971,0.6663,0.2572,0.2562,0.2394,0.2522,0.2504
FRCNN-ConvNeXt + YOLO26s,0.2477,0.2469,"raw, w=(1, 2), IoU 0.3",0.3,0.8948,0.6687,0.2533,0.2563,0.2356,0.246,0.2479
FCOS-MViT + YOLO26s,0.2444,0.2431,"rank, w=(2, 1), IoU 0.2",0.8,0.8958,0.6676,0.2495,0.2509,0.2342,0.247,0.2406
FRCNN-ConvNeXt + FCOS-MViT,0.2436,0.2421,"raw, w=(1, 2), IoU 0.3",0.5,0.8964,0.6676,0.2528,0.2508,0.2347,0.2449,0.2354
FRCNN-ConvNeXt,0.2336,0.2335,NMS 0.3,0.7,0.891,0.6571,0.2422,0.2413,0.2239,0.2325,0.2296
FCOS-MViT,0.2318,0.2317,NMS 0.2,0.45,0.8914,0.6562,0.2427,0.2362,0.2224,0.2349,0.2224
YOLO26s,0.2311,0.2306,NMS 0.2,0.1,0.886,0.6589,0.2375,0.2346,0.2187,0.2292,0.2361


The trio minus the pairs and the single models (paired bootstrap on the 26,684 images):

In [18]:
OOF_SAMPLES = np.random.default_rng(0).integers(0, len(oof_ids), (2000, len(oof_ids)), dtype=np.int32)
trio = tuple(oof)
pd.DataFrame({" + ".join(c): diff(vec[trio], vec[c], OOF_SAMPLES) for c in sorted(ens, key=lambda c: -ens[c]["score"])
              if c != trio}).T.rename_axis("trio minus")

,difference,95% interval
trio minus,,
FRCNN-ConvNeXt + YOLO26s,0.0034,"[0.0006, 0.0061]"
FCOS-MViT + YOLO26s,0.0066,"[0.0036, 0.0096]"
FRCNN-ConvNeXt + FCOS-MViT,0.0075,"[0.0036, 0.0111]"
FRCNN-ConvNeXt,0.0174,"[0.0132, 0.0215]"
FCOS-MViT,0.0193,"[0.0149, 0.0235]"
YOLO26s,0.0199,"[0.0162, 0.0234]"


**Finding:** on all 5 folds the trio is again the best: 0.2510 out-of-fold (split-half 0.2500), +0.0034 over the best pair (interval +0.0006 to +0.0061) and +0.017 to +0.020 over the single models; AUC 0.897, F1 0.666. Lower than on fold 0 alone (0.2616), as for the single models.

The chosen post-processing without TTA (section 9 adds flip TTA and saves it for 06):

In [19]:
chosen = ens[trio]
ensemble = {"models": {m: {"runs": [fold_dir(m, f).relative_to(DET).as_posix() for f in range(5)], "nms_iou": oof_best_iou[m]}
                       for m in trio},
            "wbf": chosen["setting"], "threshold": float(chosen["threshold"]), "oof_score": float(chosen["score"])}
ensemble

{'models': {'FRCNN-ConvNeXt': {'runs': ['grid_256/fasterrcnn_convnext_tiny_f0_s0',
    'cv_256/fasterrcnn_convnext_tiny_f1_s0',
    'cv_256/fasterrcnn_convnext_tiny_f2_s0',
    'cv_256/fasterrcnn_convnext_tiny_f3_s0',
    'cv_256/fasterrcnn_convnext_tiny_f4_s0'],
   'nms_iou': 0.3},
  'FCOS-MViT': {'runs': ['grid_256/fcos_mobilevitv2_150_f0_s0',
    'cv_256/fcos_mobilevitv2_150_f1_s0',
    'cv_256/fcos_mobilevitv2_150_f2_s0',
    'cv_256/fcos_mobilevitv2_150_f3_s0',
    'cv_256/fcos_mobilevitv2_150_f4_s0'],
   'nms_iou': 0.2},
  'YOLO26s': {'runs': ['yolo_256/yolo26s_f0_s0',
    'cv_256/yolo26s_f1_s0',
    'cv_256/yolo26s_f2_s0',
    'cv_256/yolo26s_f3_s0',
    'cv_256/yolo26s_f4_s0'],
   'nms_iou': 0.2}},
 'wbf': 'rank, w=(1, 1, 1), IoU 0.3',
 'threshold': 0.8,
 'oof_score': 0.2510421574115753}

## 9. Horizontal-flip TTA

Each of the 15 models (3 × 5 folds) also predicted on its validation and test images flipped left–right (`scripts/predict_flip.py`); the boxes are flipped back (x → 1024 − x − width). Per model, the original and the flipped boxes are fused by WBF (as in section 6: two inputs, equal weights, a view without a box there votes 0), after the same NMS on both. Out-of-fold score by variant and NMS IoU (*original* = section 7):

In [20]:
fold_of_id = dict(zip(oof_ids, fold_of))
oof_flip = {m: pd.concat([pd.read_csv(fold_dir(m, f) / "val_preds_flip.csv").assign(fold=f) for f in range(5)],
                         ignore_index=True) for m in CHOSEN}
VARIANTS = ["original", "flipped", "WBF 0.3", "WBF 0.4", "WBF 0.5"]


def tta(orig, flip, variant, iou):
    """NMS at iou on both views; then the flipped boxes, or both fused by WBF at the variant's IoU."""
    flip = nms_at(flip, iou)
    if variant == "flipped":
        return flip
    p = wbf([nms_at(orig, iou), flip], iou_thr=float(variant.split()[1]))
    return p.assign(fold=p["patient_id"].map(fold_of_id))


def tta_and_score(orig, flip, variant, iou):
    p = tta(orig, flip, variant, iou)
    return p, threshold_scores(p, oof_ids)


# the new variants (8 processes); "original" from section 7
tasks = [(m, v, iou) for m in oof for v in VARIANTS[1:] for iou in IOUS]
out = Parallel(n_jobs=8)(delayed(tta_and_score)(oof[m], oof_flip[m], v, iou) for m, v, iou in tasks)
tta_preds = {t: o[0] for t, o in zip(tasks, out)}
tta_scores = {t: o[1] for t, o in zip(tasks, out)} | {(m, "original", iou): oof_scores[(m, iou)] for m in oof for iou in IOUS}
tta_table = pd.Series({k: np.nanmean(s, axis=1).max() for k, s in tta_scores.items()}).unstack(2)[IOUS]
tta_table.reindex(pd.MultiIndex.from_product([list(oof), VARIANTS], names=["model", "variant"]))

own     0.5     0.4     0.3     0.2
model          variant                                         
FRCNN-ConvNeXt original  0.2291  0.2291  0.2329  0.2336  0.2333
               flipped   0.2255  0.2255  0.2304  0.2315  0.2316
               WBF 0.3   0.2304  0.2304  0.2355  0.2389  0.2387
               WBF 0.4   0.2320  0.2320  0.2381  0.2383  0.2381
               WBF 0.5   0.2335  0.2335  0.2378  0.2378  0.2375
FCOS-MViT      original  0.2207  0.2287  0.2313  0.2316  0.2318
               flipped   0.2157  0.2243  0.2269  0.2272  0.2273
               WBF 0.3   0.1851  0.1962  0.2042  0.2317  0.2339
               WBF 0.4   0.2092  0.2143  0.2324  0.2339  0.2336
               WBF 0.5   0.2190  0.2305  0.2328  0.2325  0.2322
YOLO26s        original  0.1940  0.2223  0.2278  0.2305  0.2311
               flipped   0.1952  0.2238  0.2285  0.2317  0.2323
               WBF 0.3   0.2392  0.2387  0.2370  0.2352  0.2358
               WBF 0.4   0.2389  0.2358  0.2342  0.2351  0.2356
               WBF 0.5   0.2330  0.2291  0.2319  0.2333  0.2338

**Finding:**
- The flipped images alone score about as well as the original ones (−0.0045 to +0.0012 at the best NMS).
- WBF of the two beats the original for every model: Faster R-CNN 0.2389 vs 0.2336 (WBF 0.3, NMS 0.3), FCOS 0.2339 vs 0.2318 (WBF 0.4, NMS 0.3), YOLO26s 0.2392 vs 0.2311 (WBF 0.3, own NMS).
- WBF merges overlapping boxes much like NMS does, so with it the stricter NMS matters less (YOLO26s is best with its own).

**The trio with flip TTA:** each model at its best WBF variant and NMS above, then the same search as in section 8. *without TTA* = the trio of section 8:

In [21]:
best_tta = {m: max([(v, iou) for v in VARIANTS[2:] for iou in IOUS],
                   key=lambda k: np.nanmean(tta_scores[(m, *k)], axis=1).max()) for m in oof}
inputs_tta = {}
for m, (v, iou) in best_tta.items():
    p = tta_preds[(m, v, iou)]
    inputs_tta[m] = {"raw": p, "rank": rank_scores(p, by="fold")}
ens_tta, vec_tta = search(inputs_tta, oof_ids, oof_pos)
print({m: f"{v}, NMS {iou}" for m, (v, iou) in best_tta.items()})
pd.DataFrame({"without TTA": ens[trio], "flip TTA": ens_tta}).T

{'FRCNN-ConvNeXt': 'WBF 0.3, NMS 0.3', 'FCOS-MViT': 'WBF 0.4, NMS 0.3', 'YOLO26s': 'WBF 0.3, NMS own'}


,score,split-half,setting,threshold,AUC,F1
without TTA,0.251,0.25,"rank, w=(1, 1, 1), IoU 0.3",0.8,0.8971,0.6663
flip TTA,0.2547,0.2539,"rank, w=(1, 1, 2), IoU 0.4",0.8,0.8968,0.6658


Flip TTA minus without, paired bootstrap as in section 8, and by fold:

In [22]:
pd.DataFrame({"flip TTA minus without": {**diff(vec_tta, vec[trio], OOF_SAMPLES),
              **{f"fold {f}": np.nanmean(vec_tta[fold_of == f]) - np.nanmean(vec[trio][fold_of == f]) for f in range(5)}}}).T

,difference,95% interval,fold 0,fold 1,fold 2,fold 3,fold 4
flip TTA minus without,0.0036,"[0.0012, 0.0064]",0.0013,0.0053,0.0031,0.0035,0.0049


**Finding:** flip TTA raises the trio from 0.2510 to 0.2547 (+0.0036, interval +0.0012 to +0.0064, positive in all 5 folds; split-half 0.2539 vs 0.2500). AUC and F1 stay the same (0.897, 0.666): TTA improves where the boxes are, not which images get one. The trio is used with flip TTA.

The chosen post-processing, for 06: per model, NMS at `nms_iou` on both views (*own* = none) and WBF of the two at `tta_wbf_iou`; then the trio WBF:

In [23]:
ensemble["models"] = {m: {**ensemble["models"][m], "nms_iou": iou, "tta_wbf_iou": float(v.split()[1])}
                      for m, (v, iou) in best_tta.items()}
ensemble.update(wbf=ens_tta["setting"], threshold=float(ens_tta["threshold"]), oof_score=float(ens_tta["score"]))
(DET / "ensemble_256.json").write_text(json.dumps(ensemble, indent=2))
ensemble

{'models': {'FRCNN-ConvNeXt': {'runs': ['grid_256/fasterrcnn_convnext_tiny_f0_s0',
    'cv_256/fasterrcnn_convnext_tiny_f1_s0',
    'cv_256/fasterrcnn_convnext_tiny_f2_s0',
    'cv_256/fasterrcnn_convnext_tiny_f3_s0',
    'cv_256/fasterrcnn_convnext_tiny_f4_s0'],
   'nms_iou': 0.3,
   'tta_wbf_iou': 0.3},
  'FCOS-MViT': {'runs': ['grid_256/fcos_mobilevitv2_150_f0_s0',
    'cv_256/fcos_mobilevitv2_150_f1_s0',
    'cv_256/fcos_mobilevitv2_150_f2_s0',
    'cv_256/fcos_mobilevitv2_150_f3_s0',
    'cv_256/fcos_mobilevitv2_150_f4_s0'],
   'nms_iou': 0.3,
   'tta_wbf_iou': 0.4},
  'YOLO26s': {'runs': ['yolo_256/yolo26s_f0_s0',
    'cv_256/yolo26s_f1_s0',
    'cv_256/yolo26s_f2_s0',
    'cv_256/yolo26s_f3_s0',
    'cv_256/yolo26s_f4_s0'],
   'nms_iou': 'own',
   'tta_wbf_iou': 0.3}},
 'wbf': 'rank, w=(1, 1, 2), IoU 0.4',
 'threshold': 0.8,
 'oof_score': 0.25468122959136963}

## 10. Summary

- **Detectors for 06:** Faster R-CNN + ConvNeXt-Tiny, FCOS + MobileViTv2-1.5 and YOLO26s at 256 × 256, each trained on the 5 folds, with horizontal-flip TTA; their boxes fused by WBF (rank scores, YOLO26s weight 2, IoU 0.4), box threshold 0.8. Out-of-fold score 0.2547, AUC 0.897, F1 0.666. The settings are saved in `outputs/detection/ensemble_256.json`.
- **What helped:** a stricter NMS (+0.003 to +0.043 per model except SSD, most for YOLO), WBF of the 3 detectors (+0.017 to +0.020 out-of-fold over each single model), flip TTA (+0.0036).
- **Did not help:** 384 × 384, YOLO26m, more than 3 models in the ensemble. SSD and the EfficientNet backbones were the weakest.
- **Left for 06:** gating by the classifier from 04, the box size and the final threshold, chosen on these out-of-fold boxes.
- **Limits:** one seed per run. Architecture, backbone, size and ensemble members were chosen on fold 0 alone (5,337 images), where single-model differences have intervals of about ±0.01. The bootstrap intervals cover the choice of images only, not seed noise. The out-of-fold score still contains the optimism of picking the best of 32 WBF settings (split-half 0.2539).